In [4]:
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# --- 1. CONFIGURACIÓN ---
NOMBRE_GRUPO = "G2_RealEstate_ConstanzaTorres" 
CATEGORIA_ACTUAL = "Departamentos_Arriendo_Coquimbo"

try:
    # Conexión al servicio 'database' de tu docker-compose
    client = MongoClient("database", 27017)
    db = client['ArriendosPrueba'] 
    coleccion = db['G2_ArriendosCoquimbo']
    print("1. CONEXIÓN EXITOSA: Base de datos conectada.")
except Exception as e:
    print(f"ERROR DE CONEXIÓN: {e}")

1. CONEXIÓN EXITOSA: Base de datos conectada.


In [5]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("🚀 Iniciando prueba de extracción Avanzada (con Scroll y Filtros)...")

# 1. Configuración Headless (Para entornos de servidor/Docker/Jupyter)
options = Options()
options.add_argument("--headless=new") # Modo invisible obligatorio para tu entorno
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

try:
    # 2. Navegación
    url = "https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo"
    print(f"🌐 Navegando a: {url}")
    driver.get(url)

    # 3. Esperar resultados iniciales
    print("⏳ Esperando carga inicial...")
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
    )
    
    # --- LA MAGIA DEL SCROLL ---
    print("📜 Deslizando hasta el final de la página para cargar todo...")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5) # Le damos 5 segundos para que carguen las imágenes y textos rezagados

    # 4. Capturar bloques
    bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
    print(f"✅ Se detectaron {len(bloques)} bloques en total (incluyendo posibles anuncios).\n")
    print("-" * 50)

    # 5. Iterar sobre los primeros 10 bloques para ver si el filtro funciona
    propiedades_validas = 0
    
    for bloque in bloques[:10]:
        # Si ya encontramos 5 propiedades válidas, paramos la prueba
        if propiedades_validas >= 5:
            break
            
        try:
            # Extraemos los textos primero
            nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").text
            precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").text
            
            # --- FILTRO ANTI-BLANCOS ---
            # Si el nombre o el precio están vacíos, es publicidad o cargó mal. Lo ignoramos.
            if not nombre.strip() or not precio_texto.strip():
                print("   ⚠️ [Bloque ignorado: Publicidad o datos vacíos]")
                continue
                
            # Si pasamos el filtro, es una propiedad real
            propiedades_validas += 1
            print(f"🏠 PROPIEDAD VÁLIDA #{propiedades_validas}:")
            print(f"   ➤ Título: {nombre}")
            print(f"   ➤ Precio bruto: {precio_texto}")
            
            # Prueba de limpieza
            precio_limpio = precio_texto.replace("$", "").replace(".", "").strip()
            if precio_limpio.isdigit():
                print(f"   ➤ Precio limpio (para BD): {float(precio_limpio)}")
            else:
                print(f"   ➤ Precio limpio (para BD): 0.0 (No numérico)")
                
            print("-" * 50)
            
        except Exception:
            # Si ni siquiera tiene la clase del título, definitivamente es publicidad
            print("   ⚠️ [Bloque ignorado: Estructura no coincide]")
            continue

except Exception as e:
    print(f"🚨 Error crítico durante la prueba: {e}")

finally:
    # 6. Cierre seguro
    print("🧹 Prueba finalizada. Cerrando el navegador.")
    driver.quit()

🚀 Iniciando prueba de extracción Avanzada (con Scroll y Filtros)...
🌐 Navegando a: https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo
⏳ Esperando carga inicial...
📜 Deslizando hasta el final de la página para cargar todo...
✅ Se detectaron 48 bloques en total (incluyendo posibles anuncios).

--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #1:
   ➤ Título: Arriendo Centro De Coquimbo: Depto Amoblado Con Vista Al Mar
   ➤ Precio bruto: $
450.000
   ➤ Precio limpio (para BD): 450000.0
--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #2:
   ➤ Título: Se Arrienda Departamento Marzo A Diciembre Costa Elqui
   ➤ Precio bruto: $
650.000
   ➤ Precio limpio (para BD): 650000.0
--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #3:
   ➤ Título: Edificio Manuel Jesús Rivera (1 Est. 1 Bod.) Vista Sur Pis
   ➤ Precio bruto: $
390.000
   ➤ Precio limpio (para BD): 390000.0
-------------------------------------------

In [ ]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. FORZAR LA PANTALLA PARA noVNC
# Esto le dice a Chrome en qué monitor virtual debe dibujarse para que VNC lo pueda grabar
os.environ["DISPLAY"] = ":99"  # IMPORTANTE: Si sigue negra, cambia ":0" por ":1" o ":99"

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza completada. Pantalla virtual configurada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2"
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()

# 🚨 ¡ATENCIÓN! Comentamos el "--headless=new" para que puedas verlo en noVNC 🚨
# options.add_argument("--headless=new")

# Argumentos de estabilidad obligatorios para entornos Docker/Servidor
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo")

    for nivel_pagina in range(limite_paginas):
        print(f"\n--- Procesando Página {nivel_pagina + 1} ---")

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
        )
        
        # --- SCROLL HUMANO (para que pueda tomar todos los datos ---
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) # Pausa final para que asienten los últimos bloques

        # Capturamos todos los bloques presentes en la página
        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        
        propiedades_extraidas = 0

        for i, bloque in enumerate(bloques):
            try:
                # --- LA MAGIA DEL textContent ---
                # Ahora extraemos el texto desde las entrañas del HTML, 
                # así no importa si la propiedad ya no está visible en pantalla
                nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent")
                try:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")

                # Filtro anti-blancos (por si acaso detecta un banner publicitario real)
                if not nombre or not precio_texto or not nombre.strip() or not precio_texto.strip():
                    continue

                try:
                    caracteristicas = bloque.find_element(By.CSS_SELECTOR, ".poly-component__attributes-list").get_attribute("textContent")

                except:
                    caracteristicas = "Sin datos"

                try:
                    direccion = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent")

                except:
                    direccion = "Dirección no especificada"

                datos_finales.append({
                    "identificador": nombre.strip(),
                    "valor": precio_texto,
                    "caracteristicas": caracteristicas.strip(),
                    "direccion": direccion.strip(),
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                propiedades_extraidas += 1
                
            except:
                # Si el bloque tiene otro HTML (Proyectos nuevos), lo ignora
                continue
                
        print(f"Se extrajeron {propiedades_extraidas} propiedades reales de {len(bloques)} bloques.")

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CSS_SELECTOR, ".andes-pagination__button--next a")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f"\n Extracción terminada: {len(datos_finales)} propiedades obtenidas en total.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    # Cierra el navegador
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    if datos_finales:
        client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
        db = client["ArriendosPrueba"]
        coleccion = db["G2_ArriendosCoquimbo"] 

        for d in datos_finales:
            # Limpia el valor antes de convertirlo (Quitamos $, puntos y saltos de línea)
            v_limpio = str(d["valor"]).replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print("Datos cargados en MongoDB correctamente.")
    else:
        print("No hay datos válidos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, lower, when, contains

# 1. Iniciar la Sesión de Spark configurada para MongoDB
# Forzamos el cierre de sesiones previas para evitar conflictos de memoria
try: 
    spark.stop()
except: 
    pass

spark = SparkSession.builder \
    .appName("Analisis_Inmobiliario_G2") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/ArriendosPrueba.G2_ArriendosCoquimbo") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

# 2. Cargar los datos desde MongoDB
df = spark.read.format("mongodb").load()

print(f"Éxito: Se han cargado {df.count()} registros desde la base de datos.")

# 3. Transformación de Negocio: Identificar Amenidades
# Vamos a crear columnas nuevas (Booleanas) para ver si el depto tiene piscina o gimnasio
# basándonos en el texto de las especificaciones o el identificador.

df_con_amenidades = df.withColumn("tiene_piscina", 
    when(lower(col("identificador")).contains("piscina") | 
         lower(col("caracteristicas")).contains("piscina"), True).otherwise(False)
).withColumn("tiene_gimnasio", 
    when(lower(col("identificador")).contains("gimnasio") | 
         lower(col("caracteristicas")).contains("gimnasio"), True).otherwise(False)
)

# 4. Limpieza de Outliers (Precios irreales)
# Filtramos arriendos menores a 100.000 o mayores a 5.000.000 para no sesgar el promedio
df_validado = df_con_amenidades.filter((col("valor") > 100000) & (col("valor") < 5000000))

df_validado.select("identificador", "valor", "tiene_piscina", "tiene_gimnasio").show(10, truncate=False)

Éxito: Se han cargado 144 registros desde la base de datos.
+------------------------------------------------------------+---------+-------------+--------------+
|identificador                                               |valor    |tiene_piscina|tiene_gimnasio|
+------------------------------------------------------------+---------+-------------+--------------+
|Arriendo Centro De Coquimbo: Depto Amoblado Con Vista Al Mar|450000.0 |false        |false         |
|Se Arrienda Departamento Marzo A Diciembre Costa Elqui      |650000.0 |false        |false         |
|Edificio Manuel Jesús Rivera (1 Est. 1 Bod.) Vista Sur Pis  |390000.0 |false        |false         |
|Arriendo Hermoso Penthouse A Pasos De Av Mar (148150)       |1500000.0|false        |false         |
|Arriendo Departamento Amoblado, La  Herradura               |650000.0 |false        |false         |
|Arriendo Departamento Amoblado Detras Casino Enjoy Coquimbo |630000.0 |false        |false         |
|Moderno Depto Amoblad

In [8]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_Prueba") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/ArriendosPrueba.G2_ArriendosCoquimbo") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 144 registros.
+--------------------+--------------------+--------------------+-------------------+-----+--------------------+---------+
|                 _id|     caracteristicas|           direccion|      fecha_captura|grupo|       identificador|    valor|
+--------------------+--------------------+--------------------+-------------------+-----+--------------------+---------+
|69e545bda6c5e6caf...|1 dormitorio1 bañ...|Melgarejo 902, Ce...|2026-04-19 21:13:51|   G2|Arriendo Centro D...| 450000.0|
|69e545bda6c5e6caf...|2 dormitorios2 ba...|Peñuelas Norte, 1...|2026-04-19 21:13:51|   G2|Se Arrienda Depar...| 650000.0|
|69e545bda6c5e6caf...|3 dormitorios1 ba...|Manuel Jesús Rive...|2026-04-19 21:13:51|   G2|Edificio Manuel J...| 390000.0|
|69e545bda6c5e6caf...|4 dormitorios3 ba...|Avenida Los Pesca...|2026-04-19 21:13:51|   G2|Arriendo Hermoso ...|1500000.0|
|69e545bda6c5e6caf...|2 dormitorios2 ba...|Sindempart, Coquimbo|2026-04-19 21:13:51|   G2|Arriendo Departam...| 650000

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, when, avg, round, count

# 1. Iniciar/Recuperar sesión
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_Amenidades_Coquimbo") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/ArriendosPrueba.G2_ArriendosCoquimbo") \
    .getOrCreate()

# 2. Cargar datos
df = spark.read.format("mongodb").load()

# 3. Creación de indicadores para múltiples amenidades
# Buscamos palabras clave tanto en el título como en las características
df_rico = df.withColumn(
    "piscina", when(lower(col("identificador")).contains("piscina") | lower(col("caracteristicas")).contains("piscina"), "Sí").otherwise("No")
).withColumn(
    "quincho", when(lower(col("identificador")).contains("quincho") | lower(col("caracteristicas")).contains("quincho"), "Sí").otherwise("No")
).withColumn(
    "gimnasio", when(lower(col("identificador")).contains("gimnasio") | lower(col("caracteristicas")).contains("gym") | lower(col("caracteristicas")).contains("gimnasio"), "Sí").otherwise("No")
).withColumn(
    "estacionamiento", when(lower(col("identificador")).contains("estacionamiento") | lower(col("caracteristicas")).contains("estacionamiento"), "Sí").otherwise("No")
).withColumn(
    "terraza", when(lower(col("identificador")).contains("terraza") | lower(col("caracteristicas")).contains("terraza"), "Sí").otherwise("No")
)

# 4. Limpieza de precios (Filtro de seguridad para arriendos en Coquimbo)
# Ignoramos valores menores a 200k (podrían ser solo estacionamientos) y mayores a 3M
df_final = df_rico.filter((col("valor") >= 200000) & (col("valor") <= 3000000))

print(f"Análisis listo sobre {df_final.count()} departamentos válidos.")

Análisis listo sobre 141 departamentos válidos.


In [10]:
# Reporte detallado de impacto en el precio
print("--- IMPACTO DE LAS AMENIDADES EN EL PRECIO (PROMEDIOS) ---")

amenidades = ["piscina", "quincho", "gimnasio", "estacionamiento", "terraza"]

for am in amenidades:
    print(f"\nAnálisis de: {am.upper()}")
    df_final.groupBy(am).agg(
        count("identificador").alias("Cantidad"),
        round(avg("valor"), 0).alias("Precio_Promedio")
    ).orderBy(am, ascending=False).show()

--- IMPACTO DE LAS AMENIDADES EN EL PRECIO (PROMEDIOS) ---

Análisis de: PISCINA
+-------+--------+---------------+
|piscina|Cantidad|Precio_Promedio|
+-------+--------+---------------+
|     Sí|       1|       650000.0|
|     No|     140|       627071.0|
+-------+--------+---------------+


Análisis de: QUINCHO
+-------+--------+---------------+
|quincho|Cantidad|Precio_Promedio|
+-------+--------+---------------+
|     No|     141|       627234.0|
+-------+--------+---------------+


Análisis de: GIMNASIO
+--------+--------+---------------+
|gimnasio|Cantidad|Precio_Promedio|
+--------+--------+---------------+
|      No|     141|       627234.0|
+--------+--------+---------------+


Análisis de: ESTACIONAMIENTO
+---------------+--------+---------------+
|estacionamiento|Cantidad|Precio_Promedio|
+---------------+--------+---------------+
|             Sí|       2|       550000.0|
|             No|     139|       628345.0|
+---------------+--------+---------------+


Análisis de: TER

In [12]:
from pyspark.sql.functions import col, lower, when, avg, round, count, regexp_extract

# 1. Asegurar sesión activa
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_Dimensiones_Coquimbo") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/ArriendosPrueba.G2_ArriendosCoquimbo") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

# 2. Extracción numérica de Habitaciones y Baños
# Usamos regexp_extract para sacar el número antes de 'dorm' y 'baño'
df_dimensiones = df.withColumn(
    "num_habitaciones", 
    regexp_extract(col("caracteristicas"), r"(\d+)\s+dorm", 1).cast("int")
).withColumn(
    "num_banos", 
    regexp_extract(col("caracteristicas"), r"(\d+)\s+baño", 1).cast("int")
)

# 3. Mantener también las amenidades (Piscina, etc.) para un análisis completo
df_completo = df_dimensiones.withColumn(
    "piscina", when(lower(col("caracteristicas")).contains("piscina"), "Sí").otherwise("No")
).withColumn(
    "estacionamiento", when(lower(col("caracteristicas")).contains("estacionamiento"), "Sí").otherwise("No")
)

# 4. Limpieza de Outliers (Aseguramos que no haya valores nulos en habitaciones)
df_limpio = df_completo.filter((col("valor") > 200000) & (col("num_habitaciones").isNotNull()))

print(f"Análisis estructurado sobre {df_limpio.count()} departamentos.")
df_limpio.select("identificador", "valor", "num_habitaciones", "num_banos", "piscina").show(20)

Análisis estructurado sobre 140 departamentos.
+--------------------+---------+----------------+---------+-------+
|       identificador|    valor|num_habitaciones|num_banos|piscina|
+--------------------+---------+----------------+---------+-------+
|Arriendo Centro D...| 450000.0|               1|        1|     No|
|Se Arrienda Depar...| 650000.0|               2|        2|     No|
|Edificio Manuel J...| 390000.0|               3|        1|     No|
|Arriendo Hermoso ...|1500000.0|               4|        3|     No|
|Arriendo Departam...| 650000.0|               2|        2|     No|
|Arriendo Departam...| 630000.0|               2|        2|     No|
|Moderno Depto Amo...| 850000.0|               3|        2|     No|
|Departamento Amob...| 600000.0|               2|        1|     No|
|Depto Amoblado Pr...| 580000.0|               3|        2|     No|
|Se Arrienda Depar...| 700000.0|               2|        2|     No|
|Departamento En A...| 470000.0|               3|        1|     No|
|

In [13]:
print("--- PRECIO PROMEDIO POR NÚMERO DE HABITACIONES ---")
df_limpio.groupBy("num_habitaciones").agg(
    count("identificador").alias("Cantidad_Deptos"),
    round(avg("valor"), 0).alias("Precio_Promedio")
).orderBy("num_habitaciones").show()

print("--- PRECIO PROMEDIO POR NÚMERO DE BAÑOS ---")
df_limpio.groupBy("num_banos").agg(
    count("identificador").alias("Cantidad_Deptos"),
    round(avg("valor"), 0).alias("Precio_Promedio")
).orderBy("num_banos").show()

--- PRECIO PROMEDIO POR NÚMERO DE HABITACIONES ---
+----------------+---------------+---------------+
|num_habitaciones|Cantidad_Deptos|Precio_Promedio|
+----------------+---------------+---------------+
|               1|             11|       430909.0|
|               2|             76|       575000.0|
|               3|             48|       658333.0|
|               4|              5|      1530000.0|
+----------------+---------------+---------------+

--- PRECIO PROMEDIO POR NÚMERO DE BAÑOS ---
+---------+---------------+---------------+
|num_banos|Cantidad_Deptos|Precio_Promedio|
+---------+---------------+---------------+
|        1|             54|       495556.0|
|        2|             81|       657778.0|
|        3|              4|      1600000.0|
|        4|              1|      1250000.0|
+---------+---------------+---------------+

